# M04 — Induced prism forward physics and controls

**Level 3 · Prerequisites: foundations notebook; read time: 35 minutes.**

## Objectives

Run the product induced-prism forward operator on its frozen synthetic fixture. Compare components with the independent Choclo physical oracle and distinguish linear TMI from exact magnitude anomaly. This is not susceptibility inversion or remanence classification.

## Setup

Use the already reviewed Python 3.12 pipeline environment. This notebook installs nothing.

Import repository code and confirm its source file.

In [ ]:
import runpy,sys
from pathlib import Path
import numpy as np
root=Path.cwd();sys.path.insert(0,str(root/'data-pipeline'))
import magnetic_forward
# Reuse the product's frozen synthetic physical control.
fixture=runpy.run_path(str(root/'tests/numerics/test_magnetic_forward.py'))
print(magnetic_forward.ENGINE)

## Data

Run the product induced-prism forward operator on its frozen synthetic fixture. Compare components with the independent Choclo physical oracle and distinguish linear TMI from exact magnitude anomaly. This is not susceptibility inversion or remanence classification.

**Files:** tests/numerics/test_magnetic_forward.py, docs/design/features/m04-induced-prism/algorithms.md, docs/design/features/m04-induced-prism/validation.md

## Method

For I positive down and D clockwise from north, f=(cos I sin D, cos I cos D, −sin I) in ENU; B₀=Ff. Uniform induced magnetization is M=χ·10⁻⁹B₀/μ₀ A m⁻¹. Secondary b gives linear TMI f·b; exact magnitude anomaly is |B₀+b|−F. Assumptions: uniform field, scalar susceptibility, no self-demagnetization and no remanence.

Execute the product method on the declared control.

In [ ]:
request=fixture['request']()
# Compare actual product response and independent physical prism calculation.
result=magnetic_forward.forward_magnetic(request)
oracle=fixture['physical_oracle'](request)
max_error=float(np.max(np.abs(result['field_components_nt']-oracle)))
print({'components_nT':result['field_components_nt'].tolist(),'linear_TMI_nT':result['linear_tmi_nt'].tolist(),'exact_magnitude_nT':result['exact_magnitude_anomaly_nt'].tolist(),'max_component_error_nT':max_error,'accepted':result['linearization']['full_method_accepted']})

Adversarial or negative control: isolate one assumption or violate one declared physical constraint.

In [ ]:
from copy import deepcopy
changed=deepcopy(request);changed['inducing_field']['declination_deg']=42.0
# Change only a declared field assumption; this is not field classification.
alternative=magnetic_forward.forward_magnetic(changed)
delta=float(np.max(np.abs(alternative['field_components_nt']-result['field_components_nt'])))
print({'max_component_change_nT':delta})
assert delta>0

## State of the art

SimPEG/Geoana is production; Choclo uses a separate analytic prism calculation. A changed field tests sensitivity but cannot identify the real field. Exact magnitude and linear TMI differ; the returned Jacobian applies to linear TMI only.

## Results

Use frozen independent tolerances, not rounded display values. A changed declination should change components. The operator sets field-source and full-method acceptance false; M04 survey inversion remains outside this lesson.

## Summary

Keep field convention, units, observables and independent oracle together. A forward image is not recovered geology.

## References

- https://raw.githubusercontent.com/simpeg/simpeg/v0.25.2/simpeg/potential_fields/magnetics/sources.py
- https://raw.githubusercontent.com/simpeg/simpeg/v0.25.2/simpeg/utils/mat_utils.py
- https://www.fatiando.org/choclo/v0.3.2/api/generated/choclo.prism.magnetic_field.html
- https://doi.org/10.1016/j.cageo.2015.09.015